# MovieLens 100K - Recommendation System

**Course:** Data Mining  
**Dataset:** MovieLens 100K  
**Models:** user-based CF, item-based CF, SVD, bias baseline, metadata hybrid model, optional neural CF  
**Metrics:** RMSE, MAE, Precision@K, Recall@K, HitRate@K  
**Reproducibility:** fixed `random_state=42`; main experiment uses a per-user temporal split

Run this notebook from the project root after placing the MovieLens 100K files under `data/raw/ml-100k/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform
import time
from IPython.display import display

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, FIGURES_DIR, K_NEIGHBORS
from src.data_loader import load_ratings, load_movies, dataset_summary
from src.data_cleaning import (
    align_ratings_with_movies,
    clean_movies,
    clean_ratings,
    data_quality_report,
)
from src.preprocessing import (
    split_ratings,
    time_based_split,
    user_temporal_split,
    global_mean,
)
from src.metrics import rmse, mae, evaluate_topk, catalog_coverage
from src.baselines import (
    GlobalMeanBaseline,
    UserMeanBaseline,
    ItemMeanBaseline,
    RandomRatingBaseline,
    MostPopularBaseline,
    BiasBaseline,
)
from src.analysis import (
    cold_start_error,
    genre_error,
    movie_popularity_error,
    relevant_items_by_user,
    user_activity_error,
    recommend_top_k_from_model,
    random_recommend_batch,
    recommendation_frequency,
    recommendation_diversity,
    recommendation_novelty,
    sampled_candidate_items_by_user,
    residual_bias_by_rating,
    tiered_error,
    bootstrap_rmse_ci,
    paired_error_test,
)
from src.user_based_cf import UserBasedCF
from src.item_based_cf import ItemBasedCF
from src.hybrid_model import MetadataHybridRegressor
from src import visualization as viz

try:
    from src.neural_cf import NeuralCFRecommender
    TORCH_AVAILABLE = True
except ImportError:
    NeuralCFRecommender = None
    TORCH_AVAILABLE = False

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"random_state={RANDOM_STATE}")
print(f"PyTorch available: {TORCH_AVAILABLE}")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"numpy={np.__version__}, pandas={pd.__version__}")
print("Expected runtime: about 4-8 minutes on a modern laptop CPU")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
raw_ratings = load_ratings()
raw_movies = load_movies()

cleaned_ratings = clean_ratings(raw_ratings)
cleaned_movies = clean_movies(raw_movies)
ratings, movies = align_ratings_with_movies(cleaned_ratings, cleaned_movies)

quality = data_quality_report(raw_ratings, ratings, raw_movies, movies)
print("Data quality report after explicit cleaning:")
display(quality)

summary = dataset_summary(ratings)
print("Cleaned dataset summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

ratings.head()

In [ ]:
movies[["movie_id", "title"]].head(10)

In [ ]:
viz.plot_rating_distribution(ratings)
viz.plot_ratings_per_user(ratings)
viz.plot_ratings_per_movie(ratings)
viz.plot_genre_counts(movies)
print(f"Figures saved to {FIGURES_DIR}")

## 3. Preprocessing - Train / Test Split

In [ ]:
train, test = user_temporal_split(ratings)
global_time_train, global_time_test = time_based_split(ratings)
random_train, random_test = split_ratings(ratings, random_state=RANDOM_STATE)
y_test = test["rating"].values

cold_users = set(test["user_id"]) - set(train["user_id"])
cold_items = set(test["movie_id"]) - set(train["movie_id"])

print(f"Main per-user temporal split: train={len(train):,}, test={len(test):,}")
print(f"Global chronological reference: train={len(global_time_train):,}, test={len(global_time_test):,}")
print(f"Random split kept only for sensitivity reference: train={len(random_train):,}, test={len(random_test):,}")
print(f"Cold users in main test: {len(cold_users)}")
print(f"Cold items in main test: {len(cold_items)}")
print(f"Train global mean: {global_mean(train):.4f}")

## 4. Model Training & Evaluation

All models are fit on the cleaned **per-user temporal train split** only. For every user, earlier ratings are used for training and later ratings are held out for testing. This avoids random row leakage while keeping collaborative-filtering evaluation focused on users with observed history.

In [ ]:
results = []
predictions = {}
model_timings = []

rating_baselines = [
    GlobalMeanBaseline(),
    UserMeanBaseline(),
    ItemMeanBaseline(),
    BiasBaseline(n_epochs=20, reg=1.0),
    RandomRatingBaseline(random_state=RANDOM_STATE),
]

for model in rating_baselines:
    fit_start = time.perf_counter()
    model.fit(train)
    fit_seconds = time.perf_counter() - fit_start
    pred_start = time.perf_counter()
    pred = model.predict_batch(test)
    predict_seconds = time.perf_counter() - pred_start
    name = model.__class__.__name__.replace("Baseline", "")
    predictions[name] = pred
    results.append({
        "model": name,
        "rmse": rmse(y_test, pred),
        "mae": mae(y_test, pred),
    })
    model_timings.append({
        "model": name,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
    })

pd.DataFrame(results).sort_values("rmse")

### 4.1 User-Based Collaborative Filtering

In [ ]:
user_cf = UserBasedCF(k=K_NEIGHBORS)
print("Fitting user-based CF (similarity matrix)...")
fit_start = time.perf_counter()
user_cf.fit(train)
fit_seconds = time.perf_counter() - fit_start

print("Predicting on test set...")
pred_start = time.perf_counter()
user_pred = user_cf.predict_batch(test)
predict_seconds = time.perf_counter() - pred_start
predictions["User-Based CF"] = user_pred

results.append({
    "model": "User-Based CF",
    "rmse": rmse(y_test, user_pred),
    "mae": mae(y_test, user_pred),
})
model_timings.append({
    "model": "User-Based CF",
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
})
print(f"User-Based CF - RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.2 Item-Based Collaborative Filtering

In [ ]:
item_cf = ItemBasedCF(k=K_NEIGHBORS)
print("Fitting item-based CF (similarity matrix)...")
fit_start = time.perf_counter()
item_cf.fit(train)
fit_seconds = time.perf_counter() - fit_start

print("Predicting on test set...")
pred_start = time.perf_counter()
item_pred = item_cf.predict_batch(test)
predict_seconds = time.perf_counter() - pred_start
predictions["Item-Based CF"] = item_pred

results.append({
    "model": "Item-Based CF",
    "rmse": rmse(y_test, item_pred),
    "mae": mae(y_test, item_pred),
})
model_timings.append({
    "model": "Item-Based CF",
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
})
print(f"Item-Based CF - RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.3 Optional - SVD Matrix Factorization (sklearn TruncatedSVD)

Latent-factor model using `sklearn.decomposition.TruncatedSVD` on a sparse, mean-centered user-item matrix.

In [ ]:
from src.svd_model import SVDRecommender

svd = SVDRecommender(random_state=RANDOM_STATE)
print("Fitting TruncatedSVD...")
fit_start = time.perf_counter()
svd.fit(train)
fit_seconds = time.perf_counter() - fit_start

print("Predicting on test set...")
pred_start = time.perf_counter()
svd_pred = svd.predict_batch(test)
predict_seconds = time.perf_counter() - pred_start
predictions["SVD (TruncatedSVD)"] = svd_pred

results.append({
    "model": "SVD (TruncatedSVD)",
    "rmse": rmse(y_test, svd_pred),
    "mae": mae(y_test, svd_pred),
})
model_timings.append({
    "model": "SVD (TruncatedSVD)",
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
})
print(f"SVD - RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.4 Metadata Hybrid Regressor

This model adds explicit MovieLens metadata and engineered statistics: user rating style, item popularity, genre one-hot columns, release year, and interaction features. It is a lightweight hybrid baseline before neural CF.

In [ ]:
hybrid = MetadataHybridRegressor(alpha=10.0)
print("Fitting metadata hybrid regressor...")
fit_start = time.perf_counter()
hybrid.fit(train, movies)
fit_seconds = time.perf_counter() - fit_start

pred_start = time.perf_counter()
hybrid_pred = hybrid.predict_batch(test)
predict_seconds = time.perf_counter() - pred_start
predictions["Metadata Hybrid"] = hybrid_pred

results.append({
    "model": "Metadata Hybrid",
    "rmse": rmse(y_test, hybrid_pred),
    "mae": mae(y_test, hybrid_pred),
})
model_timings.append({
    "model": "Metadata Hybrid",
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
})
print(f"Metadata Hybrid - RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.4.1 Metadata Hybrid Feature Documentation

The hybrid model is intentionally interpretable. It uses user behavior statistics, item popularity statistics, MovieLens genre indicators, release year, and two interaction features. The point is not to beat the bias baseline at all costs, but to test whether metadata improves rating prediction under a simple linear model.

In [ ]:
hybrid_features = pd.DataFrame({"feature": hybrid.feature_cols_})
print(f"Metadata Hybrid feature count: {len(hybrid_features)}")
display(hybrid_features)

### 4.5 Neural Collaborative Filtering (PyTorch)

A deep-learning recommender built with PyTorch. User and movie embeddings are concatenated and passed through a multi-layer perceptron (MLP) to model non-linear user-item interactions. It is trained end-to-end with an MSE loss and the AdamW optimizer; the per-epoch training loss is recorded and plotted as a training-process visualization. Requires PyTorch (listed in requirements.txt / environment.yml).

In [ ]:
RUN_NCF = True  # NeuralCF is a featured deep-learning model in the report.

if RUN_NCF and TORCH_AVAILABLE:
    ncf = NeuralCFRecommender(
        embedding_dim=32,
        hidden_dims=(64, 32),
        epochs=20,
        batch_size=512,
        learning_rate=1e-3,
        weight_decay=1e-5,
        random_state=RANDOM_STATE,
        device="cpu",  # CPU is more reproducible for coursework runs.
    )
    print("Fitting Neural CF...")
    ncf.fit(train, experiment_dir=str(PROJECT_ROOT / "experiments" / "neuralcf"))
    ncf_pred = ncf.predict_batch(test)
    predictions["Neural CF"] = ncf_pred
    results.append({
        "model": "Neural CF",
        "rmse": rmse(y_test, ncf_pred),
        "mae": mae(y_test, ncf_pred),
    })
    print(f"Neural CF - RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")
    print(f"Training MSE by epoch: {[round(x, 4) for x in ncf.history_]}")

    # Training-process visualization: per-epoch training loss (MSE) curve.
    epochs_axis = list(range(1, len(ncf.history_) + 1))
    plt.figure(figsize=(7, 4.2))
    plt.plot(epochs_axis, ncf.history_, marker="o", color="#00ac1c", linewidth=2, markersize=4)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss (MSE)")
    plt.title("NeuralCF Training Loss Curve (MovieLens 100K)")
    plt.grid(True, alpha=0.3)
    plt.xticks(epochs_axis)
    plt.tight_layout()
    _ncf_loss_path = Path(FIGURES_DIR) / "neuralcf_loss_curve.png"
    plt.savefig(_ncf_loss_path, dpi=150)
    plt.show()
    print(f"Saved NeuralCF loss curve to {_ncf_loss_path}")
else:
    print("Neural CF skipped. Set RUN_NCF=True and install PyTorch to run it.")



## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values("rmse")
timing_df = pd.DataFrame(model_timings)
comparison_df = results_df.merge(timing_df, on="model", how="left")
comparison_df

In [ ]:
viz.plot_model_comparison(results_df)

for name, preds in predictions.items():
    if name == "Global Mean":
        continue
    viz.plot_predicted_vs_actual(y_test, preds, name)
    viz.plot_residuals(y_test, preds, name)

print(f"All figures saved to {FIGURES_DIR}")

## 6. Top-K Ranking Evaluation

A held-out movie is treated as relevant if the user's actual test rating is at least `RELEVANCE_THRESHOLD`. The main threshold is 4.0 because ratings 4-5 usually indicate positive preference in MovieLens. The sensitivity table below also reports thresholds 3.0 and 3.5, because this binary conversion is a modeling assumption and materially affects Precision/Recall/NDCG.

In [ ]:
TOP_K = 10
MAX_TOPK_USERS = 60
RELEVANCE_THRESHOLD = 4.0
rng = np.random.default_rng(RANDOM_STATE)
available_users = np.array(sorted(test["user_id"].unique()), dtype=int)
user_ids = sorted(rng.choice(
    available_users,
    size=min(MAX_TOPK_USERS, len(available_users)),
    replace=False,
).tolist())
all_items = set(ratings["movie_id"])

test_items_per_user = test.groupby("user_id").size()
print(f"Top-N users evaluated: {len(user_ids)}")
print(f"Average held-out ratings per evaluated user: {test_items_per_user.loc[user_ids].mean():.2f}")
print(f"Users with fewer than {TOP_K} held-out ratings: {(test_items_per_user.loc[user_ids] < TOP_K).sum()}")
print(f"Relevance definition: actual held-out rating >= {RELEVANCE_THRESHOLD}")

relevant_all = relevant_items_by_user(test, threshold=RELEVANCE_THRESHOLD)
relevant = {user_id: relevant_all.get(user_id, set()) for user_id in user_ids}
ranking_rows = []
recommendation_sets = {}

random_recs = random_recommend_batch(train, user_ids, all_items, k=TOP_K, random_state=RANDOM_STATE)
recommendation_sets["Random"] = random_recs
random_metrics = evaluate_topk(random_recs, relevant, k=TOP_K)
ranking_rows.append({
    "model": "Random",
    **random_metrics,
    "catalog_coverage": catalog_coverage(list(random_recs.values()), all_items),
})

popular = MostPopularBaseline().fit(train)
popular_recs = popular.recommend_batch(user_ids, k=TOP_K)
recommendation_sets["Most Popular"] = popular_recs
popular_metrics = evaluate_topk(popular_recs, relevant, k=TOP_K)
ranking_rows.append({
    "model": "Most Popular",
    **popular_metrics,
    "catalog_coverage": catalog_coverage(list(popular_recs.values()), all_items),
})

for model_name, fitted_model in [
    ("GlobalMean", rating_baselines[0]),
    ("UserMean", rating_baselines[1]),
    ("ItemMean", rating_baselines[2]),
    ("Bias", rating_baselines[3]),
    ("RandomRating", rating_baselines[4]),
    ("Item-Based CF", item_cf),
    ("User-Based CF", user_cf),
    ("Metadata Hybrid", hybrid),
    ("SVD (TruncatedSVD)", svd),
]:
    recs = recommend_top_k_from_model(
        fitted_model,
        train=train,
        user_ids=user_ids,
        all_items=all_items,
        k=TOP_K,
        max_users=None,
    )
    recommendation_sets[model_name] = recs
    if model_name == "Bias":
        print("Bias recommendation sample:", list(recs.items())[:2])
    metrics = evaluate_topk(recs, relevant, k=TOP_K)
    ranking_rows.append({
        "model": model_name,
        **metrics,
        "catalog_coverage": catalog_coverage(list(recs.values()), all_items),
    })

ranking_df = pd.DataFrame(ranking_rows).sort_values("recall_at_k", ascending=False)
display(ranking_df.style.format({
    "precision_at_k": "{:.4f}",
    "recall_at_k": "{:.4f}",
    "hit_rate_at_k": "{:.4f}",
    "ndcg_at_k": "{:.4f}",
    "catalog_coverage": "{:.4f}",
}))
viz.plot_topn_metrics(ranking_df, filename="topn_metrics_comparison.png")

sampled_candidates = sampled_candidate_items_by_user(
    train=train,
    test=test,
    user_ids=user_ids,
    all_items=all_items,
    threshold=RELEVANCE_THRESHOLD,
    n_negatives=100,
    random_state=RANDOM_STATE,
)
sampled_relevant = relevant
sampled_ranking_rows = []
for model_name, fitted_model in [
    ("GlobalMean", rating_baselines[0]),
    ("UserMean", rating_baselines[1]),
    ("ItemMean", rating_baselines[2]),
    ("Bias", rating_baselines[3]),
    ("RandomRating", rating_baselines[4]),
    ("Item-Based CF", item_cf),
    ("User-Based CF", user_cf),
    ("Metadata Hybrid", hybrid),
    ("SVD (TruncatedSVD)", svd),
]:
    sampled_recs = recommend_top_k_from_model(
        fitted_model,
        train=train,
        user_ids=user_ids,
        all_items=all_items,
        k=TOP_K,
        max_users=None,
        candidate_items_by_user=sampled_candidates,
    )
    metrics = evaluate_topk(sampled_recs, sampled_relevant, k=TOP_K)
    sampled_ranking_rows.append({"model": model_name, **metrics})

sampled_popular_recs = {
    int(user_id): [
        item_id for item_id in popular.recommend(int(user_id), k=len(all_items))
        if item_id in set(sampled_candidates[int(user_id)])
    ][:TOP_K]
    for user_id in user_ids
}
sampled_rng = np.random.default_rng(RANDOM_STATE)
sampled_random_recs = {}
for user_id in user_ids:
    candidates = list(sampled_candidates[int(user_id)])
    sample_size = min(TOP_K, len(candidates))
    sampled_random_recs[int(user_id)] = (
        sampled_rng.choice(candidates, size=sample_size, replace=False).astype(int).tolist()
        if sample_size else []
    )
for model_name, recs in [("Most Popular", sampled_popular_recs), ("Random", sampled_random_recs)]:
    metrics = evaluate_topk(recs, sampled_relevant, k=TOP_K)
    sampled_ranking_rows.append({"model": model_name, **metrics})

sampled_ranking_df = pd.DataFrame(sampled_ranking_rows).sort_values("recall_at_k", ascending=False)
print("Sampled-negative Top-N evaluation: positives plus up to 100 sampled unseen items per user.")
display(sampled_ranking_df.style.format({
    "precision_at_k": "{:.4f}",
    "recall_at_k": "{:.4f}",
    "hit_rate_at_k": "{:.4f}",
    "ndcg_at_k": "{:.4f}",
}))

threshold_rows = []
for threshold in [3.0, 3.5, 4.0]:
    rel_all = relevant_items_by_user(test, threshold=threshold)
    rel = {user_id: rel_all.get(user_id, set()) for user_id in user_ids}
    for model_name, recs in recommendation_sets.items():
        metrics = evaluate_topk(recs, rel, k=TOP_K)
        threshold_rows.append({"threshold": threshold, "model": model_name, **metrics})
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.style.format({
    "threshold": "{:.1f}",
    "precision_at_k": "{:.4f}",
    "recall_at_k": "{:.4f}",
    "hit_rate_at_k": "{:.4f}",
    "ndcg_at_k": "{:.4f}",
}))
print(
    "Threshold note: 4.0 is used as the main threshold because it represents "
    "clearly positive MovieLens feedback. Lower thresholds are reported as "
    "sensitivity checks because they make relevance more permissive."
)

freq = recommendation_frequency(recommendation_sets["Most Popular"], all_items)
print(f"Movies never recommended by Most Popular in this sample: {(freq['recommendation_count'] == 0).sum()}")
viz.plot_recommendation_frequency(freq, filename="most_popular_recommendation_frequency.png")

diversity_rows = []
for model_name, recs in recommendation_sets.items():
    div = recommendation_diversity(recs, movies)
    nov = recommendation_novelty(recs, train)
    diversity_rows.append({
        "model": model_name,
        "genre_diversity": div["genre_diversity"].mean(),
        "unique_genres": div["unique_genres"].mean(),
        "mean_self_information": nov["mean_self_information"].mean(),
    })
diversity_df = pd.DataFrame(diversity_rows).sort_values("genre_diversity", ascending=False)
print("Diversity/novelty: higher values mean broader genre spread or less-popular recommendations.")
display(diversity_df.style.format({
    "genre_diversity": "{:.4f}",
    "unique_genres": "{:.2f}",
    "mean_self_information": "{:.4f}",
}))
viz.plot_diversity_novelty(diversity_df, filename="diversity_novelty.png")

## 6.1 Final Decision Table

This table combines rating-prediction quality, Top-N ranking quality, catalog coverage, training/prediction time, and approximate memory. `Most Popular` and `Random` are ranking-only baselines here, so RMSE/MAE are intentionally left blank for them.

In [ ]:
n_users = train["user_id"].nunique()
n_items = train["movie_id"].nunique()
memory_rows = [
    {"model": "User-Based CF", "memory_mb": n_users * n_users * 8 / 1e6},
    {"model": "Item-Based CF", "memory_mb": n_items * n_items * 8 / 1e6},
    {"model": "SVD (TruncatedSVD)", "memory_mb": (n_users * 50 + n_items * 50) * 8 / 1e6},
    {"model": "Bias", "memory_mb": (n_users + n_items) * 8 / 1e6},
    {"model": "Metadata Hybrid", "memory_mb": len(hybrid.feature_cols_) * 8 / 1e6},
    {"model": "GlobalMean", "memory_mb": 8 / 1e6},
    {"model": "UserMean", "memory_mb": n_users * 8 / 1e6},
    {"model": "ItemMean", "memory_mb": n_items * 8 / 1e6},
    {"model": "RandomRating", "memory_mb": 8 / 1e6},
    {"model": "Most Popular", "memory_mb": n_items * 8 / 1e6},
    {"model": "Random", "memory_mb": 8 / 1e6},
]
memory_df = pd.DataFrame(memory_rows)

rating_decision = comparison_df.rename(columns={
    "rmse": "RMSE",
    "mae": "MAE",
    "fit_seconds": "Train Time (s)",
    "predict_seconds": "Predict Time (s)",
})
ranking_decision = ranking_df.rename(columns={
    "precision_at_k": "Precision@10",
    "recall_at_k": "Recall@10",
    "hit_rate_at_k": "HitRate@10",
    "ndcg_at_k": "NDCG@10",
    "catalog_coverage": "Coverage",
})

decision_table = (
    pd.merge(rating_decision, ranking_decision, on="model", how="outer")
    .merge(memory_df, on="model", how="left")
    .rename(columns={"memory_mb": "Memory (MB)"})
)
for col in ["Train Time (s)", "Predict Time (s)"]:
    decision_table[col] = decision_table[col].fillna(0.0)

display(
    decision_table[[
        "model", "RMSE", "MAE", "Precision@10", "Recall@10", "NDCG@10",
        "Coverage", "Train Time (s)", "Predict Time (s)", "Memory (MB)"
    ]].sort_values(["RMSE", "Recall@10"], na_position="last")
    .style.format({
        "RMSE": "{:.4f}",
        "MAE": "{:.4f}",
        "Precision@10": "{:.4f}",
        "Recall@10": "{:.4f}",
        "NDCG@10": "{:.4f}",
        "Coverage": "{:.4f}",
        "Train Time (s)": "{:.2f}",
        "Predict Time (s)": "{:.2f}",
        "Memory (MB)": "{:.2f}",
    })
)

## 7. Diagnostic Error Analysis

The following analyses look beyond average error: cold-start cases, sparse vs active users, head vs long-tail movies, and genre-specific error.

In [ ]:
best_model_name = pd.DataFrame(results).sort_values("rmse").iloc[0]["model"]
best_pred = predictions[best_model_name]
print(f"Diagnostics using best RMSE model: {best_model_name}")

print("Cold-start error:")
display(cold_start_error(train, test, best_pred).style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}))

user_tier_rows = []
item_tier_rows = []
for model_name in ["SVD (TruncatedSVD)", "Bias", "Item-Based CF", "User-Based CF", "Metadata Hybrid"]:
    pred = predictions[model_name]
    u = tiered_error(train, test, pred, entity="user")
    u["model"] = model_name
    user_tier_rows.append(u)
    it = tiered_error(train, test, pred, entity="item")
    it["model"] = model_name
    item_tier_rows.append(it)
user_tier_df = pd.concat(user_tier_rows, ignore_index=True)
item_tier_df = pd.concat(item_tier_rows, ignore_index=True)

print("User activity tier error:")
display(user_tier_df.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}))
viz.plot_tier_error(user_tier_df, filename="user_tier_rmse.png")

print("Movie popularity tier error:")
display(item_tier_df.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}))
viz.plot_tier_error(item_tier_df, filename="item_tier_rmse.png")

print("Genre error by model (highest RMSE rows shown first):")
genre_rows = []
for model_name in ["SVD (TruncatedSVD)", "Bias", "Item-Based CF", "User-Based CF", "Metadata Hybrid"]:
    per_genre = genre_error(test, movies, predictions[model_name])
    per_genre["model"] = model_name
    genre_rows.append(per_genre)
genre_all_df = pd.concat(genre_rows, ignore_index=True)
display(
    genre_all_df.sort_values(["model", "rmse"], ascending=[True, False])
    .style.format({"rmse": "{:.4f}", "mae": "{:.4f}"})
)
genre_df = genre_all_df[genre_all_df["model"] == best_model_name].sort_values("rmse", ascending=False)
viz.plot_genre_error(genre_df, filename="genre_rmse.png")

print("Prediction bias by actual rating:")
bias_by_rating = residual_bias_by_rating(test, best_pred)
display(bias_by_rating.style.format({"mean_prediction": "{:.4f}", "mean_error": "{:.4f}", "mae": "{:.4f}"}))

tier_order = ["very_cold_0_3", "cold_4_10", "warm_11_50", "very_warm_51_plus"]
penalty_rows = []
for model_name, grouped in user_tier_df.groupby("model"):
    available = grouped.set_index("tier")
    present = [tier for tier in tier_order if tier in available.index]
    if len(present) >= 2:
        cold_tier = present[0]
        warm_tier = present[-1]
        penalty_rows.append({
            "model": model_name,
            "cold_tier": cold_tier,
            "warm_tier": warm_tier,
            "cold_minus_warm_rmse": float(
                available.loc[cold_tier, "rmse"] - available.loc[warm_tier, "rmse"]
            ),
        })
penalty_df = pd.DataFrame(
    penalty_rows,
    columns=["model", "cold_tier", "warm_tier", "cold_minus_warm_rmse"],
)
if not penalty_df.empty:
    penalty_df = penalty_df.sort_values("cold_minus_warm_rmse", ascending=False)
print("Cold-start penalty uses the coldest and warmest tiers present in this split.")
display(penalty_df.style.format({"cold_minus_warm_rmse": "{:.4f}"}))

## 8. Split Validity Check

The main split is per-user temporal. A global chronological split is stricter about absolute time but creates many cold-start test users in MovieLens 100K. The comparison below makes that trade-off explicit instead of hiding it.

In [ ]:
global_time_model = MetadataHybridRegressor(alpha=10.0).fit(global_time_train, movies)
global_time_pred = global_time_model.predict_batch(global_time_test)
global_time_y = global_time_test["rating"].to_numpy()

random_model = MetadataHybridRegressor(alpha=10.0).fit(random_train, movies)
random_pred = random_model.predict_batch(random_test)
random_y = random_test["rating"].to_numpy()

split_results = pd.DataFrame([
    {
        "split": "per_user_temporal_main",
        "test_rows": len(test),
        "cold_test_users": len(set(test["user_id"]) - set(train["user_id"])),
        "model": "Metadata Hybrid",
        "rmse": rmse(y_test, predictions["Metadata Hybrid"]),
        "mae": mae(y_test, predictions["Metadata Hybrid"]),
    },
    {
        "split": "global_chronological_reference",
        "test_rows": len(global_time_test),
        "cold_test_users": len(set(global_time_test["user_id"]) - set(global_time_train["user_id"])),
        "model": "Metadata Hybrid",
        "rmse": rmse(global_time_y, global_time_pred),
        "mae": mae(global_time_y, global_time_pred),
    },
    {
        "split": "random_holdout_reference",
        "test_rows": len(random_test),
        "cold_test_users": len(set(random_test["user_id"]) - set(random_train["user_id"])),
        "model": "Metadata Hybrid",
        "rmse": rmse(random_y, random_pred),
        "mae": mae(random_y, random_pred),
    },
])
display(split_results.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}).hide(axis="index"))

## 9. K-Fold Robustness, Learning Curves, and Scalability

These checks are lightweight by design. Full K-fold evaluation for dense UserCF/ItemCF is expensive, so this section focuses on faster models and uses deterministic samples where needed. The goal is to check robustness and overfitting without breaking laptop reproducibility.

In [ ]:
from sklearn.model_selection import KFold

print("Random-shuffle KFold below is a variance check for fast models only; the main reported experiment remains the per-user temporal split.")

cv_rows = []
cv_data = ratings.sample(n=min(12000, len(ratings)), random_state=RANDOM_STATE).reset_index(drop=True)
kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, te_idx) in enumerate(kf.split(cv_data), start=1):
    cv_train = cv_data.iloc[tr_idx].reset_index(drop=True)
    cv_test = cv_data.iloc[te_idx].reset_index(drop=True)
    cv_y = cv_test["rating"].to_numpy()
    for model_name, model in [
        ("Bias", BiasBaseline(n_epochs=15, reg=1.0)),
        ("ItemMean", ItemMeanBaseline()),
        ("Metadata Hybrid", MetadataHybridRegressor(alpha=10.0)),
    ]:
        if model_name == "Metadata Hybrid":
            model.fit(cv_train, movies)
        else:
            model.fit(cv_train)
        pred = model.predict_batch(cv_test)
        cv_rows.append({
            "fold": fold,
            "model": model_name,
            "rmse": rmse(cv_y, pred),
            "mae": mae(cv_y, pred),
        })
cv_df = pd.DataFrame(cv_rows)
cv_summary = cv_df.groupby("model").agg(
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    mae_mean=("mae", "mean"),
    mae_std=("mae", "std"),
).reset_index()
display(cv_summary.style.format({
    "rmse_mean": "{:.4f}",
    "rmse_std": "{:.4f}",
    "mae_mean": "{:.4f}",
    "mae_std": "{:.4f}",
}))

learning_rows = []
train_sample = train.sample(n=min(18000, len(train)), random_state=RANDOM_STATE).reset_index(drop=True)
test_sample = test.sample(n=min(3000, len(test)), random_state=RANDOM_STATE).reset_index(drop=True)
for frac in [0.2, 0.4, 0.6, 0.8, 1.0]:
    subset = train_sample.sample(frac=frac, random_state=RANDOM_STATE).reset_index(drop=True)
    for model_name, model in [
        ("Bias", BiasBaseline(n_epochs=15, reg=1.0)),
        ("ItemMean", ItemMeanBaseline()),
    ]:
        model.fit(subset)
        train_pred = model.predict_batch(subset)
        test_pred = model.predict_batch(test_sample)
        learning_rows.append({
            "model": model_name,
            "train_fraction": frac,
            "split": "train",
            "rmse": rmse(subset["rating"].to_numpy(), train_pred),
        })
        learning_rows.append({
            "model": model_name,
            "train_fraction": frac,
            "split": "test",
            "rmse": rmse(test_sample["rating"].to_numpy(), test_pred),
        })
learning_df = pd.DataFrame(learning_rows)
display(learning_df.head())
viz.plot_learning_curve(learning_df, filename="learning_curve.png")

n_users = train["user_id"].nunique()
n_items = train["movie_id"].nunique()
scalability_df = pd.DataFrame([
    {"model": "User-Based CF", "main_memory_object": "user-user similarity", "matrix_shape": f"{n_users} x {n_users}", "estimated_mb": n_users * n_users * 8 / 1e6},
    {"model": "Item-Based CF", "main_memory_object": "item-item similarity", "matrix_shape": f"{n_items} x {n_items}", "estimated_mb": n_items * n_items * 8 / 1e6},
    {"model": "SVD", "main_memory_object": "components + user factors", "matrix_shape": "low rank factors", "estimated_mb": (n_users * 50 + n_items * 50) * 8 / 1e6},
])
print("Approximate dense-memory footprint of main fitted objects:")
display(scalability_df.style.format({"estimated_mb": "{:.2f}"}))

## 10. Result Interpretation

The tables and figures below are generated from the current run. The conclusion should be based on these computed results rather than hard-coded historical scores.

In [ ]:
results_display = pd.DataFrame(results).sort_values("rmse")
print("=== Rating-prediction metrics on the per-user temporal test set ===")
display(
    comparison_df.sort_values("rmse").style.format({
        "rmse": "{:.4f}",
        "mae": "{:.4f}",
        "fit_seconds": "{:.2f}",
        "predict_seconds": "{:.2f}",
    }).hide(axis="index")
)

best_rating_model = results_display.iloc[0]
print(
    f"Best RMSE model in this run: {best_rating_model['model']} "
    f"(RMSE={best_rating_model['rmse']:.4f}, MAE={best_rating_model['mae']:.4f})"
)

sparsity_pct = 100.0 * summary["sparsity"]
print(
    f"\nSparsity: {summary['n_ratings']:,} ratings / "
    f"({summary['n_users']:,} users x {summary['n_movies']:,} movies) "
    f"-> {sparsity_pct:.1f}% of matrix entries are missing"
)

ci_rows = []
for model_name in ["SVD (TruncatedSVD)", "Bias", "Item-Based CF", "User-Based CF", "Metadata Hybrid"]:
    low, high = bootstrap_rmse_ci(y_test, predictions[model_name], n_bootstrap=300, random_state=RANDOM_STATE)
    ci_rows.append({"model": model_name, "rmse_ci_low": low, "rmse_ci_high": high})
ci_df = pd.DataFrame(ci_rows)
print("Bootstrap 95% confidence intervals for RMSE:")
display(ci_df.style.format({"rmse_ci_low": "{:.4f}", "rmse_ci_high": "{:.4f}"}))
print(ci_df.to_string(index=False))

paired_rows = []
for other in ["SVD (TruncatedSVD)", "Item-Based CF", "User-Based CF", "Metadata Hybrid"]:
    test_result = paired_error_test(y_test, predictions["Bias"], predictions[other])
    paired_rows.append({"comparison": f"Bias vs {other}", **test_result})
paired_df = pd.DataFrame(paired_rows)
print("Paired tests on squared errors; negative diff means Bias has lower MSE:")
display(paired_df.style.format({
    "mean_squared_error_diff": "{:.6f}",
    "t_stat": "{:.3f}",
    "p_value": "{:.4g}",
    "cohens_d_paired": "{:.4f}",
}))
print(paired_df.to_string(index=False))

In [ ]:
# Mean absolute error by true rating (Item-Based CF)
error_by_rating = (
    test.assign(pred=item_pred, abs_error=lambda d: (d["rating"] - d["pred"]).abs())
    .groupby("rating")["abs_error"]
    .mean()
    .rename("mean_abs_error")
)
print("Item-Based CF - mean absolute error by actual rating:")
display(error_by_rating.to_frame().style.format({"mean_abs_error": "{:.4f}"}))

viz.plot_error_by_rating(error_by_rating, "Item-Based CF", filename="error_by_rating_item_cf.png")
print(f"Figure saved: {FIGURES_DIR / 'error_by_rating_item_cf.png'}")

## 11. Hyperparameter Sensitivity

We vary `K_NEIGHBORS` for both user-based and item-based CF, then vary `n_components` for SVD. To keep the notebook runnable on a laptop, this sensitivity section uses a deterministic sample of the test set. Final model comparison metrics above are still computed on the full test set.

In [ ]:
SENSITIVITY_N = len(test)
sensitivity_test = test.reset_index(drop=True)
y_sens = sensitivity_test["rating"].to_numpy()
print(f"Sensitivity analysis uses the full held-out test set: {len(sensitivity_test):,} rows.")

K_VALUES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

k_rows = []
for model_name, fitted_model in [("User-Based CF", user_cf), ("Item-Based CF", item_cf)]:
    for k in K_VALUES:
        fitted_model.k = k
        pred_k = fitted_model.predict_batch(sensitivity_test)
        k_rows.append({
            "model": model_name,
            "k": k,
            "rmse": rmse(y_sens, pred_k),
            "mae": mae(y_sens, pred_k),
        })
        print(f"{model_name:13s} k={k:2d} RMSE={k_rows[-1]['rmse']:.4f} MAE={k_rows[-1]['mae']:.4f}")

k_results_df = pd.DataFrame(k_rows)
display(k_results_df.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}).hide(axis="index"))
viz.plot_k_sensitivity(k_results_df, filename="cf_k_sensitivity.png")
print(f"Figure saved: {FIGURES_DIR / 'cf_k_sensitivity.png'}")

SVD_COMPONENTS = [10, 20, 30, 40, 50, 75, 100]
svd_rows = []
for n_components in SVD_COMPONENTS:
    svd_model = SVDRecommender(n_components=n_components, random_state=RANDOM_STATE)
    svd_model.fit(train)
    pred_svd = svd_model.predict_batch(sensitivity_test)
    svd_rows.append({
        "n_components": n_components,
        "rmse": rmse(y_sens, pred_svd),
        "mae": mae(y_sens, pred_svd),
    })
    print(f"SVD n_components={n_components:3d} RMSE={svd_rows[-1]['rmse']:.4f} MAE={svd_rows[-1]['mae']:.4f}")

svd_results_df = pd.DataFrame(svd_rows)
display(svd_results_df.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}).hide(axis="index"))
viz.plot_svd_sensitivity(svd_results_df, filename="svd_components_sensitivity.png")
print(f"Figure saved: {FIGURES_DIR / 'svd_components_sensitivity.png'}")

BIAS_REG_VALUES = [0.1, 0.3, 1, 3, 10, 30, 100]
bias_rows = []
for reg in BIAS_REG_VALUES:
    bias_model = BiasBaseline(n_epochs=20, reg=reg).fit(train)
    train_eval = train.sample(n=min(10000, len(train)), random_state=RANDOM_STATE)
    train_pred = bias_model.predict_batch(train_eval)
    test_pred = bias_model.predict_batch(test)
    bias_rows.append({
        "reg": reg,
        "train_rmse_sample": rmse(train_eval["rating"].to_numpy(), train_pred),
        "test_rmse": rmse(y_test, test_pred),
        "test_mae": mae(y_test, test_pred),
    })
bias_reg_df = pd.DataFrame(bias_rows)
display(bias_reg_df.style.format({"train_rmse_sample": "{:.4f}", "test_rmse": "{:.4f}", "test_mae": "{:.4f}"}))
viz.plot_bias_regularization(
    bias_reg_df.rename(columns={"train_rmse_sample": "train_rmse"}),
    filename="bias_regularization_sensitivity.png",
)
print(f"Best bias regularization: {bias_reg_df.sort_values('test_rmse').iloc[0]['reg']}")

### How hyperparameters affect performance

A small neighborhood can be noisy because too few neighbors contribute to each prediction. A very large neighborhood can dilute local taste signals with weakly related users or items. SVD dimensionality has a similar trade-off: too few components underfit, while too many components can overfit sparse observed ratings or amplify noise.

## 12. Conclusion

This notebook implements the required recommendation pipeline with explicit data cleaning, a leakage-aware per-user temporal split, multiple baselines, user-based CF, item-based CF, SVD, metadata-aware modeling, Top-N ranking metrics, and diagnostic error analysis.

Key methodological choices are explicit: relevant Top-N items are held-out movies with actual ratings above the chosen threshold; threshold sensitivity is reported; catalog coverage is compared against Random and Most Popular baselines; and the bias baseline is tuned over regularization strengths.

The strongest RMSE model is selected from the executed results rather than hard-coded claims. Top-N ranking can favor a different model, which is expected because rating prediction and recommendation ranking optimize different objectives.

**Interpretation caveat:** Most Popular winning Recall@10/NDCG@10 is a diagnostic warning, not just a neutral trade-off. Under the full-catalog offline protocol, each user has only a small set of held-out positives and all unrated catalog items are unknown rather than true negatives. The strict relevance threshold, sparse user-item matrix, and limited overlap for nearest-neighbor CF make it hard for personalized models to accumulate enough ranking signal. The TruncatedSVD model here is also a lightweight baseline, not a fully optimized biased matrix factorization model trained with ALS or SGD.

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

best_rmse_row = results_df.sort_values("rmse").iloc[0]
global_rmse = float(results_df.loc[results_df["model"] == "GlobalMean", "rmse"].iloc[0])
improvement = (global_rmse - best_rmse_row["rmse"]) / global_rmse * 100
best_recall_row = ranking_df.sort_values("recall_at_k", ascending=False).iloc[0]
best_coverage_row = ranking_df.sort_values("catalog_coverage", ascending=False).iloc[0]
best_lambda = bias_reg_df.sort_values("test_rmse").iloc[0]["reg"]

if not penalty_df.empty:
    penalty_row = penalty_df.iloc[0]
    penalty_text = (
        f"largest cold-vs-warm RMSE gap is {penalty_row['cold_minus_warm_rmse']:.4f} "
        f"for {penalty_row['model']}"
    )
else:
    penalty_text = "no cold/warm penalty could be computed for the present tiers"

print(f"""
1. Rating Prediction:
   - Best RMSE model: {best_rmse_row['model']} (RMSE={best_rmse_row['rmse']:.4f}, MAE={best_rmse_row['mae']:.4f})
   - Improvement over GlobalMean RMSE: {improvement:.1f}%

2. Top-N Ranking:
   - Best Recall@10 model: {best_recall_row['model']} (Recall@10={best_recall_row['recall_at_k']:.4f})
   - Best catalog coverage model: {best_coverage_row['model']} (Coverage={best_coverage_row['catalog_coverage']:.4f})
   - Main relevance threshold: actual held-out rating >= {RELEVANCE_THRESHOLD}

3. Cold-Start / Sparsity:
   - Matrix sparsity: {summary['sparsity'] * 100:.1f}% missing entries
   - Cold-start penalty summary: {penalty_text}

4. Bias Regularization:
   - Best tested regularization lambda: {best_lambda}
   - Regularization controls the trade-off between train fit and test RMSE.

5. Ranking Warning:
   - Most Popular winning Recall@10/NDCG@10 suggests the personalized rankers are weak under this protocol.
   - Inspect threshold choice, sparse held-out positives, catalog coverage, and candidate generation before treating ranking results as production-ready.
""")